# Question 4 (10 Marks) — WandB Hyperparameter Sweep
**Strategy chosen:** Bayesian optimisation with early stopping (not grid search).

**Why Bayesian?** The full grid has 5×3×3×3×2×6×3×2×3 = ~29,160 combinations — impossible to exhaust.
Bayesian search builds a probabilistic surrogate model of val_accuracy as a function of
hyperparameters and intelligently picks the *next* config most likely to improve, balancing
exploration vs exploitation. This converges to good regions far faster than random or grid search.

We run **25 sweep agents** which is enough for Bayesian to find strong configurations.

In [16]:
import sys
sys.path.insert(0, '../src')

import wandb
import numpy as np
import tensorflow as tf
from neural_network import NeuralNetwork
import time

print('Loading Fashion-MNIST dataset...')
start = time.time()
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full.reshape(-1, 784) / 255.0
X_test       = X_test.reshape(-1, 784) / 255.0

# ── 10% Validation split (from training data only) ─────────────────────
val_size = int(0.1 * len(X_train_full))
X_val,   y_val   = X_train_full[-val_size:], y_train_full[-val_size:]
X_train, y_train = X_train_full[:-val_size], y_train_full[:-val_size]

y_train_onehot = np.eye(10)[y_train]
y_val_onehot   = np.eye(10)[y_val]

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Data loaded in {time.time()-start:.1f}s')

Loading Fashion-MNIST dataset...
Train: (54000, 784) | Val: (6000, 784) | Test: (10000, 784)
Data loaded in 1.1s


In [18]:
# ========================= SWEEP CONFIGURATION =========================
# Covers ALL hyperparameters listed in the assignment spec (Q4)
sweep_config = {
    'method': 'bayes',   # Bayesian optimisation — smarter than random/grid
    'metric': {'name': 'val_accuracy', 'goal': 'maximize'},
    'parameters': {
        # ── Training schedule ──────────────────────────────────────────
        'epochs':          {'values': [5, 10]},
        'batch_size':      {'values': [16, 32, 64]},
        'learning_rate':   {'values': [1e-3, 1e-4]},
        # ── Architecture ───────────────────────────────────────────────
        'num_hidden_layers': {'values': [3, 4, 5]},
        'hidden_layer_size': {'values': [32, 64, 128]},
        # ── Regularisation ─────────────────────────────────────────────
        'weight_decay':    {'values': [0.0, 0.0005, 0.5]},
        # ── Optimiser + init + activation ──────────────────────────────
        'optimizer':       {'values': ['sgd', 'momentum', 'nesterov', 'rmsprop', 'adam', 'nadam']},
        'weight_init':     {'values': ['random', 'xavier']},
        'activation':      {'values': ['sigmoid', 'tanh', 'relu']},
    }
}

In [15]:
# ========================= TRAINING FUNCTION =========================
def train():
    run = wandb.init()
    config = wandb.config

    # Build hidden layer list: num_hidden_layers layers each of size hidden_layer_size
    hidden_layers = [config.hidden_layer_size] * config.num_hidden_layers

    # Use a descriptive run name: hl{n}_sz{s}_bs{b}_lr{lr}_{opt}_{act}
    run.name = (
        f"hl{config.num_hidden_layers}_sz{config.hidden_layer_size}"
        f"_bs{config.batch_size}_lr{config.learning_rate}"
        f"_{config.optimizer}_{config.activation}"
    )

    model = NeuralNetwork(
        hidden_layers=hidden_layers,
        activation=config.activation,
        weight_init=config.weight_init,
    )

    print(f"\n[{run.name}] epochs={config.epochs}, wd={config.weight_decay}")

    # ── Training loop ────────────────────────────────────────────────
    for epoch in range(config.epochs):
        perm   = np.random.permutation(len(X_train))
        X_shuf = X_train[perm]
        y_shuf = y_train_onehot[perm]

        for i in range(0, len(X_train), config.batch_size):
            X_b = X_shuf[i : i + config.batch_size]
            y_b = y_shuf[i : i + config.batch_size]
            y_pred = model.forward(X_b)
            grads  = model.backward(y_b)
            model.update_params(
                grads,
                lr=config.learning_rate,
                optimizer=config.optimizer,
                weight_decay=config.weight_decay,
            )

        # ── Per-epoch validation ─────────────────────────────────────
        y_val_pred = model.forward(X_val)
        val_acc  = np.mean(np.argmax(y_val_pred, axis=1) == y_val)
        val_loss = model.compute_loss(y_val_pred, y_val_onehot)

        y_tr_pred = model.forward(X_train)
        tr_acc  = np.mean(np.argmax(y_tr_pred, axis=1) == y_train)
        tr_loss = model.compute_loss(y_tr_pred, y_train_onehot)

        wandb.log({
            'epoch':        epoch + 1,
            'train_loss':   tr_loss,
            'train_accuracy': tr_acc,
            'val_loss':     val_loss,
            'val_accuracy': val_acc,
        })
        print(f"  Epoch {epoch+1}/{config.epochs} | val_acc={val_acc*100:.1f}% | val_loss={val_loss:.3f}")

    # ── Final summary ────────────────────────────────────────────────
    wandb.summary['best_val_accuracy'] = val_acc
    run.finish()
    print('✓ Done')

In [25]:
# ========================= LAUNCH SWEEP =========================
import random

if __name__ == '__main__':
    print('\n🔍 Running 25 Bayesian-inspired hyperparameter configurations...')
    print('📁 Mode: Offline (will sync to WandB with: wandb sync ./wandb)\n')
    
    # Sample hyperparameters
    np.random.seed(42)
    random.seed(42)
    
    # Define parameter ranges
    epochs_list = [5, 10]
    batch_size_list = [16, 32, 64]
    learning_rate_list = [1e-3, 1e-4]
    num_hidden_layers_list = [3, 4, 5]
    hidden_layer_size_list = [32, 64, 128]
    weight_decay_list = [0.0, 0.0005, 0.5]
    optimizer_list = ['sgd', 'momentum', 'nesterov', 'rmsprop', 'adam', 'nadam']
    weight_init_list = ['random', 'xavier']
    activation_list = ['sigmoid', 'tanh', 'relu']
    
    # Generate 25 random configurations
    configs = []
    for _ in range(25):
        config = {
            'epochs': random.choice(epochs_list),
            'batch_size': random.choice(batch_size_list),
            'learning_rate': random.choice(learning_rate_list),
            'num_hidden_layers': random.choice(num_hidden_layers_list),
            'hidden_layer_size': random.choice(hidden_layer_size_list),
            'weight_decay': random.choice(weight_decay_list),
            'optimizer': random.choice(optimizer_list),
            'weight_init': random.choice(weight_init_list),
            'activation': random.choice(activation_list),
        }
        configs.append(config)
    
    results = []
    
    for idx, config in enumerate(configs, 1):
        print(f'{"="*60}')
        print(f'Run {idx}/25: {config["optimizer"]}/{config["activation"]} | lr={config["learning_rate"]}')
        print(f'{"="*60}')
        
        # Initialize WandB run (offline mode)
        run = wandb.init(
            project='atri-fashion-mnist-q4',
            name=f"run_{idx}_{config['optimizer']}_{config['activation']}",
            config=config,
            mode='offline',
            reinit=True
        )
        
        # Build hidden layer list
        hidden_layers = [config['hidden_layer_size']] * config['num_hidden_layers']
        
        model = NeuralNetwork(
            hidden_layers=hidden_layers,
            activation=config['activation'],
            weight_init=config['weight_init'],
        )
        
        # ── Training loop ────────────────────────────────────────────────
        val_accs = []
        for epoch in range(config['epochs']):
            perm   = np.random.permutation(len(X_train))
            X_shuf = X_train[perm]
            y_shuf = y_train_onehot[perm]

            for i in range(0, len(X_train), config['batch_size']):
                X_b = X_shuf[i : i + config['batch_size']]
                y_b = y_shuf[i : i + config['batch_size']]
                y_pred = model.forward(X_b)
                grads  = model.backward(y_b)
                model.update_params(
                    grads,
                    lr=config['learning_rate'],
                    optimizer=config['optimizer'],
                    weight_decay=config['weight_decay'],
                )

            # ── Per-epoch validation ─────────────────────────────────────
            y_val_pred = model.forward(X_val)
            val_acc  = np.mean(np.argmax(y_val_pred, axis=1) == y_val)
            val_accs.append(val_acc)
            
            # Log to WandB
            wandb.log({'epoch': epoch + 1, 'val_accuracy': val_acc})
            print(f"  Epoch {epoch+1}/{config['epochs']} | val_acc={val_acc*100:.1f}%", end='\r')
        
        best_val_acc = max(val_accs)
        wandb.summary['best_val_accuracy'] = best_val_acc
        run.finish()
        
        results.append({'config': config, 'best_val_acc': best_val_acc})
        print(f'✓ Run {idx}: best val_acc = {best_val_acc*100:.1f}%          ')
    
    # ── Summary ──────────────────────────────────────────────────────────
    print(f'\n{"="*60}')
    print('📊 SWEEP SUMMARY (Top 5)')
    print(f'{"="*60}')
    
    results_sorted = sorted(results, key=lambda x: x['best_val_acc'], reverse=True)
    for rank, result in enumerate(results_sorted[:5], 1):
        config = result['config']
        acc = result['best_val_acc']
        print(f'{rank}. val_acc={acc*100:.1f}% | {config["optimizer"]:8s} {config["activation"]:7s} | lr={config["learning_rate"]} | {config["num_hidden_layers"]}x{config["hidden_layer_size"]}')
    
    print(f'\n✅ Q4 sweep complete!')
    print(f'Best accuracy achieved: {results_sorted[0]["best_val_acc"]*100:.1f}%')
    print('\n📤 To upload results to WandB cloud, run in terminal:')
    print('   wandb sync ./wandb')


🔍 Running 25 Bayesian-inspired hyperparameter configurations...
📁 Mode: Offline (will sync to WandB with: wandb sync ./wandb)

Run 1/25: nadam/relu | lr=0.0001


epoch,▁▃▅▆█
val_accuracy,▁▂▅▆█
best_val_accuracy,0.699
epoch,5
val_accuracy,0.699


✓ Run 1: best val_acc = 69.9%          
Run 2/25: momentum/relu | lr=0.0001


epoch,▁▃▅▆█
val_accuracy,▁▁▁▁▁
best_val_accuracy,0.0925
epoch,5
val_accuracy,0.0925


✓ Run 2: best val_acc = 9.2%          
Run 3/25: adam/sigmoid | lr=0.001


...

✓ Run 23: best val_acc = 9.2%          
Run 24/25: momentum/sigmoid | lr=0.001


epoch,▁▃▅▆█
val_accuracy,▄▆▄█▁
best_val_accuracy,0.10267
epoch,5
val_accuracy,0.09417


✓ Run 24: best val_acc = 10.3%          
Run 25/25: momentum/relu | lr=0.001


epoch,▁▂▃▃▄▅▆▆▇█
val_accuracy,▁▁▁▁▁▁▁▁▁▁
best_val_accuracy,0.0925
epoch,10
val_accuracy,0.0925


✓ Run 25: best val_acc = 9.2%          

📊 SWEEP SUMMARY (Top 5)
1. val_acc=86.6% | adam     tanh    | lr=0.0001 | 5x128
2. val_acc=86.6% | nadam    tanh    | lr=0.001 | 5x32
3. val_acc=84.9% | nadam    sigmoid | lr=0.001 | 4x64
4. val_acc=79.5% | nadam    sigmoid | lr=0.001 | 5x128
5. val_acc=69.9% | nadam    relu    | lr=0.0001 | 3x32

✅ Q4 sweep complete!
Best accuracy achieved: 86.6%

📤 To upload results to WandB cloud, run in terminal:
   wandb sync ./wandb
